# Stage 1 — Sentiment Analysis
Compare VADER, Logistic Regression, BiLSTM, and BERT on the Yelp review sentiment task.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import get_tag_columns, make_sample_dataset, preprocess_dataframe
from sentiment.vader_model import VADERSentimentModel
from sentiment.logistic_regression import LRSentimentModel
from sentiment.lstm_model import LSTMSentimentModel
from sentiment.bert_model import BERTSentimentModel
from evaluation.metrics import (
    sentiment_report, plot_confusion_matrix,
    plot_sentiment_model_comparison, plot_train_vs_f1,
    build_comparison_table,
)

sns.set_theme(style='whitegrid')
%matplotlib inline
os.makedirs('../data/figures', exist_ok=True)

In [ ]:
# Load splits (run notebook 01 first; falls back to synthetic data)
try:
    train = pd.read_parquet('../data/train.parquet')
    val   = pd.read_parquet('../data/val.parquet')
    test  = pd.read_parquet('../data/test.parquet')
    print(f'Loaded real splits: train={len(train):,} val={len(val):,} test={len(test):,}')
except FileNotFoundError:
    print('Parquet splits not found — generating synthetic data.')
    from preprocessing import train_val_test_split
    df = preprocess_dataframe(make_sample_dataset(n=3000))
    train, val, test = train_val_test_split(df)

X_train, y_train = train['text_clean'].tolist(), train['sentiment'].tolist()
X_val,   y_val   = val['text_clean'].tolist(),   val['sentiment'].tolist()
X_test,  y_test  = test['text_clean'].tolist(),  test['sentiment'].tolist()

## Model 1: VADER (rule-based baseline)

In [ ]:
vader = VADERSentimentModel()
vader_preds, vader_infer_time = vader.timed_predict(X_test)
vader_results = sentiment_report(y_test, vader_preds, model_name='VADER')
vader_results['train_time'] = 0.0
vader_results['infer_time'] = vader_infer_time

plot_confusion_matrix(
    y_test, vader_preds, ['Negative', 'Neutral', 'Positive'],
    title='VADER Confusion Matrix',
    save_path='../data/figures/cm_vader.png'
);

## Model 2: Logistic Regression (TF-IDF)

In [ ]:
lr_model = LRSentimentModel(max_features=50_000)
lr_model.fit(X_train, y_train)
lr_preds, lr_infer_time = lr_model.timed_predict(X_test)
lr_results = sentiment_report(y_test, lr_preds, model_name='Logistic Regression')
lr_results['train_time'] = lr_model.train_time_
lr_results['infer_time'] = lr_infer_time

plot_confusion_matrix(
    y_test, lr_preds, ['Negative', 'Neutral', 'Positive'],
    title='LR Confusion Matrix',
    save_path='../data/figures/cm_lr.png'
);

print('\nTop TF-IDF tokens per class:')
for cls, tokens in lr_model.get_top_tfidf_tokens(n=10).items():
    print(f'  {cls}: {tokens}')

## Model 3: BiLSTM

In [ ]:
lstm_model = LSTMSentimentModel(embed_dim=128, hidden_dim=256, epochs=5, batch_size=128)
lstm_model.fit(X_train, y_train, val_texts=X_val, val_labels=y_val)
lstm_preds, lstm_infer_time = lstm_model.timed_predict(X_test)
lstm_results = sentiment_report(y_test, lstm_preds, model_name='BiLSTM')
lstm_results['train_time'] = lstm_model.train_time_
lstm_results['infer_time'] = lstm_infer_time

plot_confusion_matrix(
    y_test, lstm_preds, ['Negative', 'Neutral', 'Positive'],
    title='BiLSTM Confusion Matrix',
    save_path='../data/figures/cm_lstm.png'
);

## Model 4: BERT (DistilBERT fine-tuned)

In [ ]:
bert_model = BERTSentimentModel(model_name='distilbert-base-uncased', epochs=3, batch_size=32)
bert_model.fit(X_train, y_train)
bert_preds, bert_infer_time = bert_model.timed_predict(X_test)
bert_results = sentiment_report(y_test, bert_preds, model_name='BERT')
bert_results['train_time'] = bert_model.train_time_
bert_results['infer_time'] = bert_infer_time

plot_confusion_matrix(
    y_test, bert_preds, ['Negative', 'Neutral', 'Positive'],
    title='BERT Confusion Matrix',
    save_path='../data/figures/cm_bert.png'
);

## Aggregate Comparison

In [ ]:
all_results = [vader_results, lr_results, lstm_results, bert_results]

comparison_df = build_comparison_table(all_results)
print(comparison_df[['accuracy', 'f1_macro', 'f1_weighted', 'train_time', 'infer_time']].round(4))

plot_sentiment_model_comparison(all_results, save_path='../data/figures/sentiment_comparison.png');
plot_train_vs_f1(all_results, save_path='../data/figures/pareto_sentiment.png');